In [ ]:
# Example 1: basic local JSON reference resolution via replace_refs.
import jsonref

doc = {
    "definitions": {
        "address": {"type": "object", "properties": {"street": {"type": "string"}}}
    },
    "home": {"$ref": "#/definitions/address"},
    "work": {"$ref": "#/definitions/address"},
}

resolved = jsonref.replace_refs(doc)
print(resolved["home"])
print(resolved["work"])
assert resolved["home"]["type"] == "object"
assert resolved["work"]["properties"]["street"]["type"] == "string"


In [ ]:
# Example 2: jsonref.loads parses a JSON string with $ref resolution.
import jsonref

json_str = '''{
    "definitions": {"name": {"type": "string", "maxLength": 50}},
    "first_name": {"$ref": "#/definitions/name"},
    "last_name": {"$ref": "#/definitions/name"}
}'''

result = jsonref.loads(json_str)
print(result["first_name"])
print(result["last_name"])
assert result["first_name"]["type"] == "string"
assert result["last_name"]["maxLength"] == 50


In [ ]:
# Example 3: jsonref.dumps round-trips a resolved doc back to JSON with $refs preserved.
import json

import jsonref

doc = {
    "definitions": {"point": {"x": 0, "y": 0}},
    "origin": {"$ref": "#/definitions/point"},
}

resolved = jsonref.replace_refs(doc)
dumped = jsonref.dumps(resolved)
parsed_back = json.loads(dumped)
print(dumped)
assert "$ref" in dumped
assert parsed_back["origin"]["$ref"] == "#/definitions/point"


In [ ]:
# Example 4: lazy_load=False eagerly resolves refs into plain dict structures.
import jsonref

doc = {
    "definitions": {"color": {"r": 255, "g": 0, "b": 0}},
    "primary": {"$ref": "#/definitions/color"},
}

eager = jsonref.replace_refs(doc, lazy_load=False)
lazy = jsonref.replace_refs(doc, lazy_load=True)

print("eager primary:", eager["primary"])
print("lazy primary:", dict(lazy["primary"]))
assert eager["primary"]["r"] == 255
assert lazy["primary"]["g"] == 0


In [ ]:
# Example 5: nested $ref chains resolve transitively, and JsonRef is exposed as a class.
import jsonref

doc = {
    "definitions": {
        "a": {"value": 1},
        "b": {"$ref": "#/definitions/a"},
        "c": {"$ref": "#/definitions/b"},
    },
    "top": {"$ref": "#/definitions/c"},
}

resolved = jsonref.replace_refs(doc)
print("top:", resolved["top"])
print("JsonRef class:", jsonref.JsonRef)
assert resolved["top"]["value"] == 1
assert hasattr(jsonref, "JsonRef")
assert hasattr(jsonref, "replace_refs")
